# Binance NEARUSDT Futures — Real-Time Local Order Book

Connects to the Binance USDM Futures diff depth WebSocket stream for **NEARUSDT** at 100 ms granularity, maintains a fully synchronised local order book, and continuously prints Level 1, 5, and 10 spread metrics to stdout.

**Dependencies:**
```
pip install websockets requests
```

## Imports and Configuration

Standard-library and third-party imports. `Decimal` precision is set to 28 to avoid float rounding during spread calculations. Constants define the target symbol, WebSocket and REST endpoints, queue depth, and the two format strings used for tabular output.

In [25]:
import asyncio
import json
import signal
import sys
import time
from datetime import datetime, timezone, timedelta
from decimal import Decimal, InvalidOperation, getcontext
from typing import Dict, List, Tuple, Optional

import requests
import websockets


# Higher precision helps avoid avoidable float rounding during spread calculation.
getcontext().prec = 28

SYMBOL = "NEARUSDT"
STREAM_SYMBOL = SYMBOL.lower()
WS_URL = f"wss://fstream.binance.com/ws/{STREAM_SYMBOL}@depth@100ms"
REST_URL = "https://fapi.binance.com/fapi/v1/depth"
SNAPSHOT_LIMIT = 1000
QUEUE_MAXSIZE = 10000
UTC_PLUS_8 = timezone(timedelta(hours=8))

PRINT_HEADER = ("{:<23} | {:<12} | {:<12} | {:<12} | {:<12} | {:<12} | {:<12}")
PRINT_ROW = ("{:<23} | {:<12.8f} | {:<12.8f} | {:<12.8f} | {:<12.8f} | {:<12.8f} | {:<12.8f}")


PriceBook = Dict[Decimal, Decimal]
DepthEvent = dict

---

## LocalOrderBook

Holds the full local bid and ask books as `Dict[Decimal, Decimal]` mappings of price to quantity.

- **`load_snapshot`** — seeds the book from the REST depth snapshot and records `lastUpdateId`.
- **`apply_event`** — applies one `depthUpdate` diff event. Enforces the Binance continuity invariant (`pu == last_update_id`); returns `False` if a sequence gap is detected, signalling that a full resync is needed. Zero-quantity updates delete price levels; non-zero updates are absolute replacements.
- **`top_bids` / `top_asks`** — return the N best price levels sorted descending / ascending.
- **`metrics`** — computes percentage spread `(ask - bid) / mid * 100` independently at levels 1, 5, and 10, plus best bid, best ask, and mid price. Returns `None` until at least 10 levels exist on both sides.

In [26]:
class LocalOrderBook:
    def __init__(self) -> None:
        self.bids: PriceBook = {}
        self.asks: PriceBook = {}
        self.last_update_id: Optional[int] = None
        self.synced = False

    def load_snapshot(self, snapshot: dict) -> None:
        self.bids = self._levels_to_book(snapshot.get("bids", []))
        self.asks = self._levels_to_book(snapshot.get("asks", []))
        self.last_update_id = int(snapshot["lastUpdateId"])
        self.synced = True

    @staticmethod
    def _levels_to_book(levels: List[List[str]]) -> PriceBook:
        book: PriceBook = {}
        for price_str, qty_str in levels:
            try:
                price = Decimal(price_str)
                qty = Decimal(qty_str)
            except (InvalidOperation, ValueError):
                continue
            if qty != 0:
                book[price] = qty
        return book

    def apply_event(self, event: DepthEvent, check_previous: bool = True) -> bool:
        """
        Apply one Binance depthUpdate event.
        Returns False when the event sequence is broken and a full resync is required.
        """
        if self.last_update_id is None:
            return False

        first_update_id = int(event["U"])
        final_update_id = int(event["u"])
        previous_final_update_id = int(event.get("pu", -1))

        # Ignore stale events already covered by the local book.
        if final_update_id < self.last_update_id:
            return True

        # After initial sync, Binance requires pu to equal the previous event's u.
        if check_previous and previous_final_update_id != self.last_update_id:
            return False

        # Defensive check: if an event starts too far ahead, the book has a gap.
        if not check_previous and not (first_update_id <= self.last_update_id <= final_update_id):
            return False

        self._apply_side_updates(self.bids, event.get("b", []))
        self._apply_side_updates(self.asks, event.get("a", []))
        self.last_update_id = final_update_id
        return True

    @staticmethod
    def _apply_side_updates(book: PriceBook, updates: List[List[str]]) -> None:
        for price_str, qty_str in updates:
            price = Decimal(price_str)
            qty = Decimal(qty_str)
            if qty == 0:
                # Binance says removing a non-existing level can happen and is normal.
                book.pop(price, None)
            else:
                # Quantity is absolute, not a delta.
                book[price] = qty

    def top_bids(self, n: int) -> List[Tuple[Decimal, Decimal]]:
        return sorted(self.bids.items(), key=lambda item: item[0], reverse=True)[:n]

    def top_asks(self, n: int) -> List[Tuple[Decimal, Decimal]]:
        return sorted(self.asks.items(), key=lambda item: item[0])[:n]
    # Task 3 — Calculate and Print Spreads
    def metrics(self) -> Optional[Tuple[Decimal, Decimal, Decimal, Decimal, Decimal, Decimal]]:
        bids = self.top_bids(10)
        asks = self.top_asks(10)
        if len(bids) < 10 or len(asks) < 10:
            return None

        spreads = []
        for level in (1, 5, 10):
            bid_price = bids[level - 1][0]
            ask_price = asks[level - 1][0]
            mid_price = (ask_price + bid_price) / Decimal("2")
            if mid_price <= 0:
                return None
            spread_pct = (ask_price - bid_price) / mid_price * Decimal("100")
            spreads.append(spread_pct)

        best_bid = bids[0][0]
        best_ask = asks[0][0]
        best_mid = (best_bid + best_ask) / Decimal("2")
        return spreads[0], spreads[1], spreads[2], best_bid, best_ask, best_mid

---

## Output Formatting Utilities

- **`utc8_timestamp_ms`** — returns the current wall-clock time in UTC+8 as a 23-character string (`YYYY-MM-DD HH:MM:SS.mmm`) matching the table column width.
- **`fetch_snapshot`** — synchronous REST call to `GET /fapi/v1/depth` with a 10-second timeout; used inside `asyncio.to_thread` so it does not block the event loop.
- **`print_header`** — writes the column header and separator line once at startup.
- **`print_metrics`** — calls `orderbook.metrics()` and formats the result into one tab-aligned row using `PRINT_ROW`.

In [27]:
def utc8_timestamp_ms() -> str:
    now = datetime.now(UTC_PLUS_8)
    # Format: YYYY-MM-DD HH:MM:SS.mmm, length 23, matching the assessment's table width.
    return now.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]


def fetch_snapshot() -> dict:
    response = requests.get(
        REST_URL,
        params={"symbol": SYMBOL, "limit": SNAPSHOT_LIMIT},
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


def print_header() -> None:
    print(
        PRINT_HEADER.format(
            "Timestamp(UTC+8)",
            "Spread L1%",
            "Spread L5%",
            "Spread L10%",
            "Best Bid",
            "Best Ask",
            "Mid Price",
        )
    )
    print("-" * 110)


def print_metrics(orderbook: LocalOrderBook) -> None:
    data = orderbook.metrics()
    if data is None:
        return
    spread_1, spread_5, spread_10, best_bid, best_ask, best_mid = data
    print(
        PRINT_ROW.format(
            utc8_timestamp_ms(),
            float(spread_1),
            float(spread_5),
            float(spread_10),
            float(best_bid),
            float(best_ask),
            float(best_mid),
        ),
        flush=True,
    )

---

## WebSocket Receiver

`websocket_receiver` is a long-running coroutine that maintains a persistent connection to the Binance futures diff depth stream (`@depth@100ms`). It deserialises incoming messages, filters for `depthUpdate` events, and enqueues them for the processor. On any connection error it logs a warning and reconnects with exponential back-off (1 s to 30 s cap).

In [28]:
async def websocket_receiver(queue: asyncio.Queue, stop_event: asyncio.Event) -> None:
    """Continuously receive depth updates and put them into the queue."""
    backoff_seconds = 1
    while not stop_event.is_set():
        try:
            async with websockets.connect(
                WS_URL,
                ping_interval=20,
                ping_timeout=20,
                max_queue=None,
                close_timeout=5,
            ) as ws:
                backoff_seconds = 1
                async for message in ws:
                    if stop_event.is_set():
                        break
                    event = json.loads(message)
                    if event.get("e") == "depthUpdate":
                        await queue.put(event)
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            print(f"[WARN] WebSocket disconnected: {exc}. Reconnecting...", file=sys.stderr)
            await asyncio.sleep(backoff_seconds)
            backoff_seconds = min(backoff_seconds * 2, 30)

---

## Order Book Initialisation and Sync Procedure

Implements the Binance recommended sync procedure to safely bridge the REST snapshot and the live diff stream:

1. The WebSocket receiver is already running and buffering events in the queue.
2. Fetch a fresh REST depth snapshot via `asyncio.to_thread`.
3. Discard any buffered events where `u < lastUpdateId` (already covered by the snapshot).
4. Find the first event where `U <= lastUpdateId <= u` — the bridging event that overlaps with the snapshot boundary.
5. Apply the bridging event (skipping the `pu` continuity check for that first event only), then apply all subsequent buffered events normally.
6. If no bridging event is found the snapshot is too old; sleep 200 ms and retry from step 2.

`drain_queue` non-blockingly drains all currently buffered events from the queue at once.

In [29]:
async def drain_queue(queue: asyncio.Queue) -> List[DepthEvent]:
    events: List[DepthEvent] = []
    while True:
        try:
            events.append(queue.get_nowait())
        except asyncio.QueueEmpty:
            break
    return events


async def initialise_orderbook(orderbook: LocalOrderBook, queue: asyncio.Queue) -> None:
    """
    Binance sync procedure:
    1. WebSocket receiver is already running and buffering events.
    2. Fetch REST depth snapshot.
    3. Drop stale events where u < lastUpdateId.
    4. Start with the first event where U <= lastUpdateId <= u.
    """
    while True:
        snapshot = await asyncio.to_thread(fetch_snapshot)
        orderbook.load_snapshot(snapshot)
        last_update_id = orderbook.last_update_id
        assert last_update_id is not None

        buffered = await drain_queue(queue)
        buffered = [event for event in buffered if int(event["u"]) >= last_update_id]

        start_index = None
        for index, event in enumerate(buffered):
            if int(event["U"]) <= last_update_id <= int(event["u"]):
                start_index = index
                break

        if start_index is None:
            # No bridging event yet. Keep buffering and try again shortly.
            await asyncio.sleep(0.2)
            continue

        first = True
        for event in buffered[start_index:]:
            ok = orderbook.apply_event(event, check_previous=not first)
            first = False
            if not ok:
                break
        else:
            return

        await asyncio.sleep(0.2)

---

## Processor and Main Entry Point

**`orderbook_processor`** — drives the main update loop. Calls `initialise_orderbook` to sync the book, then consumes events from the queue one by one: stale events are skipped, a gap triggers reinitialisation, and a valid update triggers `print_metrics`. Unhandled exceptions restart the loop after a 1-second pause.

**`main`** — wires up the shared `asyncio.Queue` and `stop_event`, registers `SIGINT`/`SIGTERM` handlers (best-effort on Windows), launches the receiver and processor as concurrent tasks, and waits for the stop signal. The `finally` block cancels both tasks and waits for them to finish before exiting cleanly.

In [30]:
async def orderbook_processor(queue: asyncio.Queue, stop_event: asyncio.Event) -> None:
    orderbook = LocalOrderBook()
    print_header()

    while not stop_event.is_set():
        try:
            await initialise_orderbook(orderbook, queue)
            print(f"[INFO] Local order book synced at update id {orderbook.last_update_id}", file=sys.stderr)

            while not stop_event.is_set():
                event = await queue.get()

                if orderbook.last_update_id is None:
                    break

                # Stale event.
                if int(event["u"]) < orderbook.last_update_id:
                    continue

                ok = orderbook.apply_event(event, check_previous=True)
                if not ok:
                    print("[WARN] Update gap detected. Reinitialising local order book...", file=sys.stderr)
                    break

                print_metrics(orderbook)

        except asyncio.CancelledError:
            raise
        except Exception as exc:
            print(f"[WARN] Processor error: {exc}. Reinitialising...", file=sys.stderr)
            await asyncio.sleep(1)


async def main() -> None:
    queue: asyncio.Queue = asyncio.Queue(maxsize=QUEUE_MAXSIZE)
    stop_event = asyncio.Event()

    loop = asyncio.get_running_loop()
    for sig in (signal.SIGINT, signal.SIGTERM):
        try:
            loop.add_signal_handler(sig, stop_event.set)
        except NotImplementedError:
            # Windows event loop may not support signal handlers.
            pass

    receiver_task = asyncio.create_task(websocket_receiver(queue, stop_event))
    processor_task = asyncio.create_task(orderbook_processor(queue, stop_event))

    try:
        await stop_event.wait()
    finally:
        receiver_task.cancel()
        processor_task.cancel()
        await asyncio.gather(receiver_task, processor_task, return_exceptions=True)
        print("\n[INFO] Stopped by user.", file=sys.stderr)



In [31]:
if __name__ == "__main__":
    try:
        await main()
    except KeyboardInterrupt:
        print("\n[INFO] Stopped by keyboard interrupt.", file=sys.stderr)

Timestamp(UTC+8)        | Spread L1%   | Spread L5%   | Spread L10%  | Best Bid     | Best Ask     | Mid Price   
--------------------------------------------------------------------------------------------------------------
2026-05-18 16:23:29.783 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  


[INFO] Local order book synced at update id 10573213628109


2026-05-18 16:23:29.951 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:30.205 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:30.364 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:30.534 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:30.702 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:30.884 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:30.997 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:31.309 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.48500000   | 1.48450000  
2026-05-18 16:23:31.434 | 0.06736275   | 0.60626474   | 1.27989222   | 1.48400000   | 1.


[INFO] Stopped by user.


CancelledError: 